In [2]:
# Install required external packages
!pip install pandas numpy matplotlib scikit-learn

In [3]:
import os
import glob
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit
import warnings

In [4]:
# 1. Get list of CSV files (modify folder name if needed)
data_folder = os.path.join(os.getcwd(), "../data")
csv_files = glob.glob(os.path.join(data_folder, "smart_meter_data_*.csv"))
print("Found files:", csv_files)

# 2. Load each CSV into a DataFrame
dataframes = [pd.read_csv(file) for file in csv_files]


Found files: ['c:\\Users\\stros\\Documents\\Mycroft\\Incremental-Learning-MycroftMinds\\notebooks\\../data\\smart_meter_data_0.csv', 'c:\\Users\\stros\\Documents\\Mycroft\\Incremental-Learning-MycroftMinds\\notebooks\\../data\\smart_meter_data_1.csv', 'c:\\Users\\stros\\Documents\\Mycroft\\Incremental-Learning-MycroftMinds\\notebooks\\../data\\smart_meter_data_2.csv', 'c:\\Users\\stros\\Documents\\Mycroft\\Incremental-Learning-MycroftMinds\\notebooks\\../data\\smart_meter_data_3.csv']


In [6]:
import pandas as pd

def preprocess_df(dataframes):
    """
    Preprocess a list of DataFrames:
      - Print missing values per column.
      - Convert the 'time' column to datetime with UTC conversion.
      - Remove timezone info, sort the DataFrame by 'time' and reset the index.
      - Drop the 'id' column if present.
      - Create new time-related features: hour, minute, second, day, and month.
    """
    processed = []
    for idx, df in enumerate(dataframes, start=1):
        print(f"\nDataFrame {idx}:")
        
        # Print missing value percentages and column names.
        missing_pct = df.isnull().mean() * 100
        print("Percentage of missing values per column:")
        print(missing_pct)
        print("Columns in DataFrame:")
        print(df.columns.tolist())
        
        # Convert 'time' column to datetime with UTC conversion.
        df['time'] = pd.to_datetime(df['time'], errors='coerce', utc=True)
        if df['time'].isnull().any():
            print("Warning: Some 'time' values could not be converted and are NaT!")
        # Remove timezone information (naive datetime)
        df['time'] = df['time'].dt.tz_convert(None)
        df.sort_values('time', inplace=True)
        df.reset_index(drop=True, inplace=True)
        
        # Drop the 'id' column if it exists.
        if 'id' in df.columns:
            df = df.drop(columns=['id'])
        
        # Create time-based features.
        df['hour'] = df['time'].dt.hour
        df['minute'] = df['time'].dt.minute
        df['second'] = df['time'].dt.second
        df['day'] = df['time'].dt.day
        df['month'] = df['time'].dt.month
        
        processed.append(df)
    return processed

def concat_dataframes_by_time(dataframes):
    """
    Concatenate a list of DataFrames into a single DataFrame and order the rows by 'time'.
    """
    combined_df = pd.concat(dataframes, ignore_index=True)
    combined_df['time'] = pd.to_datetime(combined_df['time'], errors='coerce')
    combined_df.sort_values('time', inplace=True)
    combined_df.reset_index(drop=True, inplace=True)
    return combined_df

def get_complete_time_series(df):
    """
    Group the concatenated DataFrame by device_id and serial_number,
    then return a list of groups (DataFrames) that have no missing values.
    """
    complete_series = []
    grouped = df.groupby(['device_id', 'serial_number'])
    for (device, serial), group in grouped:
        # Ensure the group is sorted by time.
        group_sorted = group.sort_values('time')
        # Skip groups with any missing data.
        if group_sorted.isnull().any().any():
            print(f"Skipping time series for device_id: {device}, serial_number: {serial} due to missing data.")
            continue
        complete_series.append(group_sorted)
    return complete_series

def add_lag_features(df, columns, lag_freqs={'lag_1h':'1H', 
                                             'lag_12h':'12H', 
                                             'lag_24h':'24H', 
                                             'lag_1w':'7D'}):
    # add 48h lag
    # 12h lag does not give info
    


    """
    Add lag features for the specified columns based on time differences.
    - First, adds a simple one-row lag.
    - Then, adds time-based lags using pd.merge_asof for given lag frequencies.
    """
    # Create a copy of the DataFrame to avoid modifying the original data.
    df = df.copy()
    
    # Ensure that the 'time' column is converted to datetime format.
    # If there are errors during conversion, they are coerced to NaT (Not a Time).
    df['time'] = pd.to_datetime(df['time'], errors='coerce')
    
    # Sort the DataFrame by the 'time' column in ascending order.
    df.sort_values('time', inplace=True)
    
    # ---------------------------
    # Simple one-row lag: Shift each specified column by 1 row.
    # This creates a feature representing the previous value for each column.
    for col in columns:
        df[f'{col}_lag_1'] = df[col].shift(1)
    
    # ---------------------------
    # Time-based lags using pd.merge_asof:
    # For each defined lag frequency (e.g., 1 hour, 12 hours, etc.), create lagged features.
    for suffix, lag_str in lag_freqs.items():
        # Convert the lag string (e.g., '1H') to a Timedelta object.
        lag_delta = pd.Timedelta(lag_str)
        
        # Create a new column 'time_shift_{suffix}' by subtracting the lag delta from the 'time' column.
        # This column represents the target time we want to look back from.
        df[f'time_shift_{suffix}'] = df['time'] - lag_delta
        
        # Create a temporary DataFrame that includes the original 'time' and the columns for which we want lag features.
        temp = df[['time'] + columns].copy()
        
        # Rename the columns in the temporary DataFrame to include the lag suffix.
        # This helps differentiate the lagged features from the original ones.
        rename_dict = {col: f'{col}_{suffix}' for col in columns}
        temp.rename(columns=rename_dict, inplace=True)
        
        # Use pd.merge_asof to merge the original DataFrame with the temporary DataFrame containing lagged values.
        # This performs an as-of merge, aligning rows based on the nearest (backward) time match.
        merged = pd.merge_asof(
            # Sort the original DataFrame by the shifted time column (e.g., "time_shift_lag_1h").
            # This column represents the 'time' values shifted back by the lag interval.
            df.sort_values(f'time_shift_{suffix}'),
            
            # Sort the temporary DataFrame by its 'time' column.
            # This DataFrame holds the lagged values with new column names.
            temp.sort_values('time'),
            
            # Specify the left key for merging: the shifted time column in the original DataFrame.
            left_on=f'time_shift_{suffix}',
            
            # Specify the right key for merging: the actual 'time' column in the temporary DataFrame.
            right_on='time',
            
            # Set the merge direction to 'backward' so that for each row in df, the merge returns the most recent row in temp
            # that does not exceed the shifted time.
            direction='backward',
            
            # Set a tolerance to ensure that only rows within the specified lag interval (converted from lag_str) are merged.
            tolerance=pd.Timedelta(lag_str)  # Only merge if the time difference is within the lag window.
        )
        
        # For each specified column, extract the corresponding lagged value from the merged DataFrame
        # and assign it back to the original DataFrame with the appropriate suffix.
        for col in columns:
            df[f'{col}_{suffix}'] = merged[f'{col}_{suffix}'].values
        
        # Remove the temporary shifted time column from the original DataFrame as it's no longer needed.
        df.drop(columns=[f'time_shift_{suffix}'], inplace=True)
    
    # Return the DataFrame augmented with both simple and time-based lag features.
    return df

# ---------------------------
# Main Processing Workflow
# ---------------------------
#
# 1. Preprocess each DataFrame.
processed_dfs = preprocess_df(dataframes)

# 2. Concatenate the processed DataFrames by time.
data_concat = concat_dataframes_by_time(processed_dfs)

# 3. Drop the column with 100% missing values (e.g., 'values_state').
data_concat.drop(columns='values_state', inplace=True, errors='ignore')

# 4. Group the data by device_id and serial_number, extracting only complete time series.
complete_time_series = get_complete_time_series(data_concat)
print(f"Number of complete time series: {len(complete_time_series)}")

# 5. Define the columns for which to add lag features.
columns_to_add_lag = [
    'total_power_active', 'total_power_reactive', 'total_power_apparent',
    'import_energy_active', 'export_energy_active',
    'import_energy_reactive', 'export_energy_reactive',
    'total_energy_active', 'total_energy_reactive'
]

# 6. For each complete time series, apply lag feature engineering.
processed_time_series = []
for ts in complete_time_series:
    ts_lagged = add_lag_features(ts, columns_to_add_lag)
    processed_time_series.append(ts_lagged)

print(f"Processed {len(processed_time_series)} time series with lag features.")

# Optionally, concatenate all processed time series into a single DataFrame for further analysis.
data_final = pd.concat(processed_time_series).reset_index(drop=True)
data_final.sort_values(by=['device_id', 'serial_number', 'time'], inplace=True)

print("Final data processing complete. Ready for feature engineering.")



DataFrame 1:
Percentage of missing values per column:
time                        0.0
device_id                   0.0
serial_number               0.0
values_state              100.0
total_power_active          0.0
total_power_reactive        0.0
total_power_apparent        0.0
import_energy_active        0.0
export_energy_active        0.0
import_energy_reactive      0.0
export_energy_reactive      0.0
total_energy_active         0.0
total_energy_reactive       0.0
dtype: float64
Columns in DataFrame:
['time', 'device_id', 'serial_number', 'values_state', 'total_power_active', 'total_power_reactive', 'total_power_apparent', 'import_energy_active', 'export_energy_active', 'import_energy_reactive', 'export_energy_reactive', 'total_energy_active', 'total_energy_reactive']

DataFrame 2:
Percentage of missing values per column:
time                        0.0
device_id                   0.0
serial_number               0.0
values_state              100.0
total_power_active          0.0
tota

C:\Users\stros\AppData\Local\Temp\ipykernel_21572\1642898307.py:108: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  lag_delta = pd.Timedelta(lag_str)
C:\Users\stros\AppData\Local\Temp\ipykernel_21572\1642898307.py:144: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  tolerance=pd.Timedelta(lag_str)  # Only merge if the time difference is within the lag window.
C:\Users\stros\AppData\Local\Temp\ipykernel_21572\1642898307.py:108: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  lag_delta = pd.Timedelta(lag_str)
C:\Users\stros\AppData\Local\Temp\ipykernel_21572\1642898307.py:144: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  tolerance=pd.Timedelta(lag_str)  # Only merge if the time difference is within the lag window.
C:\Users\stros\AppData\Local\Temp\ipyk

Processed 39 time series with lag features.
Final data processing complete. Ready for feature engineering.


In [9]:
for ts in processed_time_series:
    print(ts)

                           time                             device_id  \
34      2023-12-31 23:01:00.000  114b44b0-0dc4-45d6-a4ac-83223ea9b2d8   
75      2023-12-31 23:06:00.094  114b44b0-0dc4-45d6-a4ac-83223ea9b2d8   
116     2023-12-31 23:11:00.274  114b44b0-0dc4-45d6-a4ac-83223ea9b2d8   
157     2023-12-31 23:16:00.551  114b44b0-0dc4-45d6-a4ac-83223ea9b2d8   
198     2023-12-31 23:20:59.565  114b44b0-0dc4-45d6-a4ac-83223ea9b2d8   
...                         ...                                   ...   
4257440 2024-12-31 22:35:36.182  114b44b0-0dc4-45d6-a4ac-83223ea9b2d8   
4257481 2024-12-31 22:40:36.128  114b44b0-0dc4-45d6-a4ac-83223ea9b2d8   
4257522 2024-12-31 22:45:36.141  114b44b0-0dc4-45d6-a4ac-83223ea9b2d8   
4257563 2024-12-31 22:50:36.175  114b44b0-0dc4-45d6-a4ac-83223ea9b2d8   
4257604 2024-12-31 22:55:36.198  114b44b0-0dc4-45d6-a4ac-83223ea9b2d8   

                                serial_number  total_power_active  \
34       960bf15f-a847-49de-9648-45db048f3cf8         

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

def create_supervised_data_from_series(df, input_features, target_feature, window_size=3):
    """
    Create a supervised learning dataset from a time series DataFrame.
    For each time step t (starting at t = window_size), the input X is created by 
    flattening the values of the specified input_features from times t-window_size to t-1,
    and the target y is the value of target_feature at time t.
    
    Parameters:
        df (pd.DataFrame): Time series data (assumed sorted by time).
        input_features (list): List of feature column names to include as inputs.
        target_feature (str): Column name of the target variable.
        window_size (int): Number of past observations to include.
    
    Returns:
        X (np.ndarray): 2D array with shape (n_samples, window_size * n_features).
        y (np.ndarray): 1D array with the target values.
    """
    X, y = [], []
    for i in range(window_size, len(df)):
        window_data = df.iloc[i-window_size:i][input_features].values.flatten()
        X.append(window_data)
        y.append(df.iloc[i][target_feature])
    return np.array(X), np.array(y)


# ---------------------------
# Parameters for the experiment
# ---------------------------
window_size = 12                         # Sliding window length (number of past observations)
target_feature = 'total_power_active'    # The target variable to forecast

# Define how many time series are available for training vs. testing.
# In our case, we train on several (to avoid overfitting) and hold one out for incremental updates.
n_series_train = 2    # number of time series used entirely for initial training
# The test series will be used for incremental prediction updates.
# For the test series, only a small seed is used to start (e.g., 1% of its supervised data)
test_training_ratio = 0.01   # use first 1% of the test series' supervised data as seed

# Incremental update parameters for the test series:
batch_size = 15      # number of supervised samples per incremental update
num_batches = 10     # number of incremental batches to process

# ---------------------------
# Assume processed_time_series is a preprocessed list of complete time series DataFrames.
# For demonstration, we assume it exists. In practice, this would come from your earlier preprocessing.
#
# For example:
#   processed_time_series = [ts1_df, ts2_df, ts3_df, ...]
#
# We split them into training series and one test series.
# ---------------------------
if len(processed_time_series) < n_series_train + 1:
    raise ValueError("Not enough time series available for training and testing.")

# Use the first n_series_train for full training and the last one as the test series.
training_series = processed_time_series[:n_series_train]
test_series = processed_time_series[-1]  # held-out series for incremental prediction

# Identify input features (exclude time and identifier columns)
all_columns = training_series[0].columns.tolist()
input_features = [col for col in all_columns if col not in ['time', 'device_id', 'serial_number']]

# ---------------------------
# Prepare the supervised dataset from the training time series
# ---------------------------
X_train_list, y_train_list = [], []
for ts in training_series:
    ts = ts.sort_values('time')  # ensure chronological order
    X_ts, y_ts = create_supervised_data_from_series(ts, input_features, target_feature, window_size)
    # Remove any rows with NaNs (if any)
    valid = ~np.isnan(X_ts).any(axis=1)
    X_train_list.append(X_ts[valid])
    y_train_list.append(y_ts[valid])

# Combine training data from all series
X_train_full = np.vstack(X_train_list)
y_train_full = np.hstack(y_train_list)
print(f"Combined training set: {X_train_full.shape[0]} samples.")

# ---------------------------
# Prepare the supervised dataset from the test series
# ---------------------------
test_series = test_series.sort_values('time')
X_test_all, y_test_all = create_supervised_data_from_series(test_series, input_features, target_feature, window_size)
valid = ~np.isnan(X_test_all).any(axis=1)
X_test_all = X_test_all[valid]
y_test_all = y_test_all[valid]

total_test_samples = len(X_test_all)
n_seed = max(1, int(test_training_ratio * total_test_samples))
print(f"Test series: Total supervised samples = {total_test_samples}, Seed samples = {n_seed}")

# ---------------------------
# Initial Model Training
# ---------------------------
# We train the initial model on all available training series (full data) plus a small seed from the test series.
# This avoids overfitting on one time series while still giving the model a start on the test series.
X_seed = X_test_all[:n_seed]
y_seed = y_test_all[:n_seed]

# Combine the seed with the full training data.
X_initial = np.vstack([X_train_full, X_seed])
y_initial = np.hstack([y_train_full, y_seed])

# Initialize the elastic (incremental) model with warm_start enabled.
model = GradientBoostingRegressor(random_state=42, warm_start=True)
model.fit(X_initial, y_initial)
print("Initial model training complete.")

# ---------------------------
# Incremental Predictions and Model Updates on Test Series
# ---------------------------
# We now process the test series in incremental batches.
# At each step, only data up to that point (and not future data) is used for training.
metrics_history = []  # store evaluation metrics for each batch

# Start from the next point after the seed
current_index = n_seed

for batch in range(num_batches):
    batch_start = current_index
    batch_end = batch_start + batch_size

    # If not enough samples remain, break out of the loop.
    if batch_end > total_test_samples:
        print(f"Not enough samples for batch {batch+1}. Ending incremental updates.")
        break

    # Current batch (new data for prediction)
    X_batch = X_test_all[batch_start:batch_end]
    y_batch = y_test_all[batch_start:batch_end]

    # Predict on the current batch
    y_pred_batch = model.predict(X_batch)

    # Calculate evaluation metrics: RMSE, MAE, and MAPE.
    rmse = np.sqrt(np.mean((y_batch - y_pred_batch) ** 2))
    mae = np.mean(np.abs(y_batch - y_pred_batch))
    mape = np.mean(np.abs((y_batch - y_pred_batch) / (np.abs(y_batch) + 1e-8))) * 100

    print(f"Test Series - Batch {batch+1}: RMSE = {rmse:.4f}, MAE = {mae:.4f}, MAPE = {mape:.2f}%")
    metrics_history.append((batch+1, rmse, mae, mape))

    # Plot the predictions vs. actual values for this batch.
    plt.figure(figsize=(10, 6))
    plt.plot(range(batch_start, batch_end), y_batch, label="Actual", marker='o')
    plt.plot(range(batch_start, batch_end), y_pred_batch, label="Predicted", marker='x', linestyle='--')
    plt.title(f"Test Series - Batch {batch+1}: Predictions vs Actual")
    plt.xlabel("Supervised Sample Index")
    plt.ylabel(target_feature)
    plt.legend()
    plt.show()

    # Update the model: incorporate the current batch into training.
    # IMPORTANT: We update only with past data. By incrementing current_index after each batch,
    # we ensure that only data available until now (seed + previous batches + current batch) is used.
    X_update = X_test_all[batch_start:batch_end]
    y_update = y_test_all[batch_start:batch_end]
    
    # Increase the number of estimators (trees) to allow for incremental learning.
    # Here, we add a fixed number (e.g. 15) new trees per batch.
    additional_trees = 15
    model.n_estimators += additional_trees
    
    # Warm-start update: fit on the new data only.
    model.fit(X_update, y_update)
    
    # Advance the index pointer.
    current_index = batch_end

# (Optional) Convert metrics_history to a DataFrame for further analysis.
metrics_df = pd.DataFrame(metrics_history, columns=["Batch", "RMSE", "MAE", "MAPE"])
print("Incremental update metrics:\n", metrics_df)


Combined training set: 202700 samples.
Test series: Total supervised samples = 101424, Seed samples = 1014
